# E2: does structure say which pairs interact?

The pooled prior draws every pair of a given order from one distribution, so it
carries no information about which pair matters. This asks whether a deposited
backbone does.

Features per measured pair: Cbeta distance, sequence separation, contact
neighbour count, the magnitude and sign agreement of the two single effects, and
hydropathy and volume shifts. A gradient-boosted model trains on every other
protein and predicts the held-out one.

Produced by `scripts/run_covariate_prior.py`. SSMuLA only, because it ships a
structure per protein.

In [1]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd().parents[1] / "src"))

pd.set_option("display.width", 180)
ROOT = Path.cwd().parents[1] / "notebooks" / "iclr" / "output_260821_e2"


## Mapping sites onto structures

SSMuLA numbers its mutable sites 1..k inside the mutable-site sequence and keeps
the residue numbers in the spec. Reading the sequence positions as residue
numbers would silently select the wrong residues, so the spec supplies the map.

In [2]:
coverage = pd.read_csv(ROOT / "structure_coverage.csv")
coverage.to_string(index=False)

'dataset protein       pdb  mapped     source_positions  min_cb_distance  max_cb_distance\n  ParD2    ParD ParD2.pdb    True         [61, 64, 80]             5.60            24.57\n  ParD3    ParD ParD3.pdb    True         [61, 64, 80]             5.76            25.07\n    GB1     GB1   GB1.pdb    True     [39, 40, 41, 54]             4.77             8.41\n   DHFR    DHFR  DHFR.pdb    True         [26, 27, 28]             5.36             7.45\n     T7      T7    T7.pdb   False      [748, 756, 758]              NaN              NaN\n    TEV     TEV   TEV.pdb    True [146, 148, 167, 170]             5.86            11.05\n TrpB3A    TrpB  TrpB.pdb    True      [104, 105, 106]             5.49             6.43\n TrpB3B    TrpB  TrpB.pdb    True      [105, 106, 107]             4.50             7.09\n TrpB3C    TrpB  TrpB.pdb    True      [106, 107, 108]             4.50             7.11\n TrpB3D    TrpB  TrpB.pdb    True      [117, 118, 119]             5.26             7.19\n TrpB3E  

## Held-out prediction

`r2_pooled_mean` is what the current prior effectively does, which is to predict
every held-out coefficient with the pooled mean.

In [3]:
results = pd.read_csv(ROOT / "holdout_r2.csv")
results.round(3).to_string(index=False)

'protein  spearman_within_landscape  n_landscapes  n_train  n_test  r2_covariates  r2_pooled_mean  spearman\n   DHFR                      0.388             1    18605    1083          0.076          -4.412     0.388\n    GB1                      0.728             1    17597    2091         -0.330          -4.112     0.728\n   ParD                     -0.029             2    17543    2145         -0.035          -5.776     0.110\n     T7                      0.393             1    19214     474         -0.133          -5.214     0.393\n    TEV                      0.554             1    17522    2166          0.494          -4.377     0.554\n   TrpB                      0.265            10     7959   11729         -0.203          -1.597     0.703'

In [4]:
print(json.dumps(json.loads((ROOT / "summary.json").read_text()), indent=2))

{
  "seed": 20260821,
  "n_pairs": 19688,
  "n_proteins": 6,
  "structures_mapped": 15,
  "median_r2_covariates": -0.08400884590582025,
  "median_r2_pooled_mean": -4.394365533947421,
  "proteins_beating_pooled_mean": 6,
  "median_spearman_pooled": 0.47376943217117284,
  "median_spearman_within_landscape": 0.390746305525661
}


## Reading the result

Two things separate.

Rank transfers. The covariates order a held-out protein's interactions at a
median Spearman near 0.39 within a single landscape, where the pooled prior can
only assign one distribution to every pair and so has no ordering at all.

Scale does not. R2 stays near zero or negative, so the model gets the ordering
roughly right and the magnitude wrong. That is the same cross-landscape
normalisation weakness the gate notebook found when the singles interquartile
range collapsed on narrow assays.

The within-landscape column matters. Several libraries mutate adjacent residues,
so every pair is in contact and distance cannot discriminate inside them. TrpB
falls from 0.703 pooled to 0.265 within a landscape, which is exactly that
effect. GB1 and TEV hold up, so the signal is real but uneven.

## Which features carry the signal

In [5]:
table = pd.read_csv(ROOT / "pair_features.csv")
features = ["cb_distance", "sequence_separation", "neighbours_min",
            "effect_product", "effect_min", "effect_max",
            "same_sign", "hydropathy_shift", "volume_shift"]
table[features + ["abs_beta"]].corr(method="spearman")["abs_beta"].drop("abs_beta").sort_values().round(3)

cb_distance           -0.361
sequence_separation   -0.210
hydropathy_shift      -0.062
volume_shift           0.238
same_sign              0.381
neighbours_min         0.625
effect_max             0.749
effect_product         0.798
effect_min             0.883
Name: abs_beta, dtype: float64

Correlation of each feature with interaction magnitude, pooled over every measured pair.